# Notebook 01 - Collecte par batch et conservation du brut

## Objectif

Dans ce notebook, on construit la **couche brute**.

On veut montrer une règle très importante :

> avant de transformer, on conserve ce qu'on a reçu.

Le staging brut joue le rôle de "boîte noire d'avion" et qu'il ne faut pas supprimer trop tôt la donnée brute. 

## Ce qu'on va produire

Deux objets :

- `ops_batch_control` : une table de pilotage des batchs
- `raw_notice` : une table brute contenant le payload JSON complet

## Pourquoi on garde le payload JSON complet

Parce que dans la vraie vie :

- un parseur peut être faux,
- un champ source peut changer,
- une règle métier peut évoluer,
- un audit peut demander la preuve de ce qui a été reçu.

Si on ne garde que la version déjà aplatie, on perd cette capacité de reprise.

In [ ]:
# Nous importons la bibliothèque json pour lire les fichiers batchs.
import json

# Nous importons hashlib pour calculer une empreinte stable du payload.
import hashlib

# Nous importons pandas pour manipuler les tables.
import pandas as pd

# Nous importons la bibliothèque pathlib pour manipuler les chemins de fichiers proprement.
from pathlib import Path

In [2]:
# Nous définissons un chemin de départ qui correspond au dossier courant du notebook.
current_dir = Path.cwd().resolve()

# Si le notebook est exécuté depuis le dossier du projet, ce test sera vrai.
if (current_dir / "data").exists():
    # Dans ce cas, le dossier courant est déjà la racine du projet.
    project_root = current_dir
else:
    # Sinon, on suppose que le notebook est exécuté depuis le dossier notebooks.
    project_root = current_dir.parent

# Nous définissons ici le dossier qui contient les lots bruts.
data_dir = project_root / "data" / "raw_batches"

# Nous définissons ici le dossier de sortie.
output_dir = project_root / "outputs"

# Nous affichons la racine retenue pour éviter toute ambiguïté.
print("project_root =", project_root)

# Nous affichons la liste des fichiers bruts disponibles.
print("raw files =", sorted(p.name for p in data_dir.glob("*.json")))

project_root = D:\Ens-EPSI\boamp_etl_dwh_tiny_explained_project
raw files = ['B20150308_01_initial.json', 'B20150309_01_incremental.json', 'B20150309_01_rerun.json']


## Étape 1 - Lister les batchs disponibles

On commence toujours par regarder ce que l'on a reçu :

- combien de batchs,
- quels types de batchs,
- quelles dates d'extraction.

Ici, on a volontairement trois cas :

- un batch initial,
- un batch incrémental,
- un rerun.

In [5]:
# Nous créons une liste ordonnée de tous les fichiers JSON de batch.
batch_files = sorted(data_dir.glob("*.json"))

# Nous affichons les noms pour contrôle visuel.
[p.name for p in batch_files]

['B20150308_01_initial.json',
 'B20150309_01_incremental.json',
 'B20150309_01_rerun.json']

## Étape 2 - Définir une petite fonction de lecture

In [6]:
# Nous définissons une fonction simple pour charger un batch JSON.
def load_batch_file(path: Path) -> dict:
    """
    Lire un fichier batch JSON et retourner son contenu sous forme de dictionnaire.

    Paramètres
    ----------
    path : Path
        Chemin du fichier batch.

    Retour
    ------
    dict
        Contenu JSON du batch.
    """

    # Nous lisons le texte du fichier en UTF-8.
    text = path.read_text(encoding="utf-8")

    # Nous convertissons ce texte JSON en dictionnaire Python.
    batch = json.loads(text)

    # Nous retournons le dictionnaire obtenu.
    return batch

## Étape 3 - Construire la table de contrôle des batchs

Cette table répond à des questions d'exploitation :

- quel batch a été reçu,
- quand,
- combien de records il contenait,
- quel était son type,
- s'est-il bien terminé.

Dans un vrai projet, cette table vit souvent dans une base d'exploitation ou une table d'audit.

In [7]:
# Nous créons une liste vide qui recevra une ligne par batch.
batch_rows = []

# Nous parcourons tous les fichiers batch un par un.
for batch_path in batch_files:

    # Nous chargeons le batch courant.
    batch = load_batch_file(batch_path)

    # Nous ajoutons une ligne de synthèse pour ce batch.
    batch_rows.append(
        {
            # batch_id sert d'identifiant métier du lot reçu.
            "batch_id": batch["batch_id"],

            # batch_type permet de distinguer INITIAL, INCREMENTAL et RERUN.
            "batch_type": batch["batch_type"],

            # source_system documente d'où vient le flux.
            "source_system": batch["source_system"],

            # extraction_ts est converti en timestamp UTC pour éviter les ambiguïtés.
            "extraction_ts": pd.to_datetime(batch["extraction_ts"], utc=True),

            # received_records donne le volume reçu.
            "received_records": len(batch["records"]),

            # status est ici fixé à SUCCESS car nos fichiers de démonstration sont lisibles.
            "status": "SUCCESS",
        }
    )

# Nous transformons la liste en DataFrame pandas.
ops_batch_control = pd.DataFrame(batch_rows)

# Nous trions la table par date d'extraction.
ops_batch_control = ops_batch_control.sort_values("extraction_ts").reset_index(drop=True)

# Nous affichons la table obtenue.
ops_batch_control


,batch_id,batch_type,source_system,extraction_ts,received_records,status
0,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,4,SUCCESS
1,B20150309_01_incremental,INCREMENTAL,BOAMP_API,2015-03-09 05:30:00+00:00,3,SUCCESS
2,B20150309_01_rerun,RERUN,BOAMP_API,2015-03-09 06:00:00+00:00,3,SUCCESS


## Étape 4 - Construire la table brute `raw_notice`

Ici, on crée **une ligne brute par record reçu**.

- on garde le `payload_json`,
- on calcule un `record_hash` pour aider à l'idempotence et à la détection de doublons.


In [8]:
# Nous créons une liste vide qui recevra une ligne brute par avis.
raw_rows = []

# Nous parcourons tous les fichiers batch.
for batch_path in batch_files:

    # Nous rechargeons le batch.
    batch = load_batch_file(batch_path)

    # Nous parcourons tous les records du batch avec un compteur de position.
    for page_no, record in enumerate(batch["records"], start=1):

        # Nous sérialisons le record en JSON texte trié pour avoir une représentation stable.
        payload_json = json.dumps(record, ensure_ascii=False, sort_keys=True)

        # Nous calculons une empreinte SHA-256 du payload.
        record_hash = hashlib.sha256(payload_json.encode("utf-8")).hexdigest()

        # Nous ajoutons la ligne brute.
        raw_rows.append(
            {
                # batch_id permet de relier le record à son lot.
                "batch_id": batch["batch_id"],

                # batch_type garde l'information INITIAL / INCREMENTAL / RERUN.
                "batch_type": batch["batch_type"],

                # source_system documente la provenance.
                "source_system": batch["source_system"],

                # extraction_ts garde la date de collecte du lot.
                "extraction_ts": pd.to_datetime(batch["extraction_ts"], utc=True),

                # page_no n'est pas une vraie pagination API ici, mais une position utile de démonstration.
                "page_no": page_no,

                # record_hash sert de clé de déduplication technique potentielle.
                "record_hash": record_hash,

                # payload_json conserve la vérité source complète.
                "payload_json": payload_json,
            }
        )

# Nous convertissons la liste en DataFrame.
raw_notice = pd.DataFrame(raw_rows)

# Nous trions pour avoir une lecture stable.
raw_notice = raw_notice.sort_values(["extraction_ts", "page_no"]).reset_index(drop=True)

# Nous affichons quelques lignes.
raw_notice.head()

,batch_id,batch_type,source_system,extraction_ts,page_no,record_hash,payload_json
0,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,1,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,"{""code_departement"": [""35""], ""code_departement..."
1,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,2,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,"{""code_departement"": [""35""], ""code_departement..."
2,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,3,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,"{""code_departement"": [""44""], ""code_departement..."
3,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,4,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,"{""code_departement"": [""44""], ""code_departement..."
4,B20150309_01_incremental,INCREMENTAL,BOAMP_API,2015-03-09 05:30:00+00:00,1,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,"{""code_departement"": [""69""], ""code_departement..."


## Étape 5 - Sauvegarder les sorties de la couche brute

On écrit les CSV pour que le notebook suivant puisse repartir de ces résultats.

Dans un vrai projet, on écrirait plutôt dans un data lake ou des fichiers parquet.

In [ ]:
# Nous définissons le dossier de sortie de la couche staging.
staging_dir = output_dir / "staging"

# Nous créons le dossier s'il n'existe pas.
staging_dir.mkdir(parents=True, exist_ok=True)

# Nous écrivons la table de contrôle des batchs.
ops_batch_control.to_csv(staging_dir / "ops_batch_control.csv", index=False)

# Nous écrivons la table brute des avis.
raw_notice.to_csv(staging_dir / "raw_notice.csv", index=False)

# Nous affichons les fichiers écrits.
sorted(p.name for p in staging_dir.glob("*.csv"))

['ops_batch_control.csv',
 'raw_notice.csv',
 'stg_notice_cpv.csv',
 'stg_notice_std.csv',
 'stg_raw_notice.csv',
 'stg_reject_notice.csv']

## Ce qu'il faut retenir

À ce stade :

- on sait **quels lots** ont été reçus,
- on sait **combien de records** chaque lot contenait,
- on possède **le payload complet** de chaque record.

On n'a pas encore fait de logique métier.  

Avant de bien modéliser, il faut d'abord **sécuriser la réception**.
